# Workplace Assistant with Claude Code

This notebook walks through the **Workplace Assistant** demo in NeMo Gym — an agentic evaluation environment where a model must complete realistic office tasks by calling tools across five workplace systems.

By the end you'll understand:
- What the environment contains and what it tests
- How the architecture fits together
- What the config files look like and what each field does
- The shape of the data — one full example entry
- How to swap in a different model (NVIDIA inference API, vLLM, or OpenAI)
- What a real agent rollout looks like, step by step
- How to extend the setup: richer verifiers, different agent harnesses, different environments

---
## 1. What is the Workplace Assistant?

The Workplace Assistant is a **demo environment** for training and evaluating models on tool use. The core problem it addresses: models need to reliably use tools to interact with structured systems — reading and writing records across email, calendar, analytics, project management, and CRM databases.

An agent is given a natural-language instruction like *"Reply to Carlos's last email about the prototype report"* and must figure out:
- which tool to call (e.g. `search_emails`, then `reply_email`)
- in what order
- what arguments to pass

A **verifier** scores the attempt by checking the **final state of the databases**, not the sequence of tool calls. This means the model is free to reach the correct outcome by any valid path — there is no single "right" order enforced. Only the end state matters.

Reward is **binary**: 1.0 if the database ends up in the correct state, 0.0 otherwise.

The dataset has **1,260 tasks** and is available on HuggingFace. Tasks range from single tool calls to multi-step chains that require looking something up before acting:

| Example task | Toolkits involved |
|---|---|
| Reply to carlos's last email about 'Task Update on Develop prototype for report generation' with 'Thanks for the update - I will get back to you tomorrow.' | Company Directory, Email |
| Change the name of the last event on December 1 to Risk Management Forum | Calendar |
| Raj is taking over all of Akira's leads that are interested in software — reassign them in the CRM | Customer Relationship Manager |

---
## 2. The Data Format

The dataset is available on HuggingFace at [`nvidia/Nemotron-RL-agent-workplace_assistant`](https://huggingface.co/datasets/nvidia/Nemotron-RL-agent-workplace_assistant).

### Downloading the dataset

The repo includes a preprocessing script that downloads the dataset and converts it into NeMo Gym's JSONL format:

```bash
# Download and preprocess (outputs to resources_servers/workplace_assistant/data/)
python resources_servers/workplace_assistant/dataset_preprocess.py --split validation
```

Or download directly with the HuggingFace `datasets` library:

```python
from datasets import load_dataset
dataset = load_dataset("nvidia/Nemotron-RL-agent-workplace_assistant", split="validation")
```

The preprocessing script handles converting the raw HuggingFace rows into the Responses API format expected by Gym — adding tool schemas, building the system prompt with the date, wrapping `verifier_metadata`. Use the script rather than loading raw rows if you want to run `gym eval run`.

---

### Data format

Each task is one JSON object with two top-level keys:

```
{
  "responses_create_params": { ... }   ← what gets sent to the agent
  "verifier_metadata":       { ... }   ← what the verifier uses to score
}
```

**`responses_create_params`** contains the full agent input:

```json
{
  "input": [
    { "role": "system",  "content": "Today's date is Thursday, 2023-11-30..." },
    { "role": "user",    "content": "Reply to carlos's last email about '...'" }
  ],
  "tools": [
    {
      "type": "function",
      "name": "email_search_emails",
      "description": "Searches for emails matching the given query...",
      "parameters": { "type": "object", "properties": { "query": { "type": "string" } } }
    }
    // ... 26 more tool schemas
  ],
  "tool_choice": "auto",
  "temperature": 1.0
}
```

**`verifier_metadata`** contains the expected outcome and task metadata:

```json
{
  "ground_truth": [
    {
      "email_reply_email": "{\"email_id\": \"00000057\", \"body\": \"Thanks for the update...\"}"
    }
  ],
  "category": "email",
  "environment_name": "workplace_assistant"
}
```

| Field | Purpose |
|---|---|
| `input` | System prompt (date/time context) + user task sent to the agent |
| `tools` | All 27 tool schemas — the agent sees these as callable functions on every turn |
| `ground_truth` | Expected tool call + arguments. The verifier checks the resulting **database state**, not whether this exact call was made. |
| `category` | Which toolkit the task tests (`email`, `calendar`, `crm`, etc.) — useful for breaking down results by domain |
| `environment_name` | Which resources server to route this task to |

**Note on `ground_truth`:** The verifier checks the *database state* that results from the tool call, not whether the model called exactly that tool with exactly those arguments. A model that found the same email ID a different way and called `reply_email` correctly would still score 1.0.


In [ ]:
import subprocess, os

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))

# Downloads the dataset from HuggingFace and writes
# resources_servers/workplace_assistant/data/validation.jsonl
result = subprocess.run(
    ["uv", "run", "python",
     "resources_servers/workplace_assistant/dataset_preprocess.py",
     "--split", "validation"],
    cwd=repo_root, capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

---
## 3. Architecture

The system has three servers that talk to each other during a rollout. NeMo Gym orchestrates them with Ray.

```
┌─────────────────────────────────────────────────────────────────────┐
│                         gym eval run                                │
│                          Orchestrator                               │
└───────────────────────────────┬─────────────────────────────────────┘
                                │ POST /run (task + tools)
                                ▼
┌───────────────────────────────────────────────────────────────────────────┐
│                     Claude Code Agent Server                              │
│                                                                           │
│   ┌──────────────────────────┐   spawn   ┌──────────────────────────┐    │
│   │    Claude Code Agent     │ ────────► │       claude CLI          │    │
│   │  (responses_api_agents/  │           │  claude -p               │    │
│   │   claude_code_agent)     │ ◄──────── │  --output-format         │    │
│   │                          │  events   │  stream-json             │    │
│   └──────┬───────────────────┘           └────────────┬─────────────┘    │
│          │ POST /seed_session                         │ MCP over HTTP    │
│          │ POST /verify                               │ (tool calls)     │
└──────────┼─────────────────────────────────────────── ┼──────────────────┘
           │                                            │
           ▼                                            ▼
┌───────────────────────────────────────────────────────────────────────────┐
│                  Workplace Assistant Resources Server                     │
│                                                                           │
│   ┌──────────────────┐    ┌────────────────────────┐    ┌─────────────┐  │
│   │  Session Manager │    │      27 MCP Tools       │    │  Verifier   │  │
│   │  /seed_session   │    │  email · calendar       │    │  /verify    │  │
│   │  (per-task state)│    │  analytics · proj mgmt  │    │             │  │
│   │                  │    │  CRM · directory         │    │  reward     │  │
│   └──────────────────┘    └────────────────────────┘    └──────┬──────┘  │
└──────────────────────────────────────────────────────────────── ┼ ────────┘
                                                                  │
                                                                  ▼
                                                            reward (0 or 1)
                                                         back to gym eval run
```

**What each piece does:**

- **`gym eval run`** — reads tasks from a JSONL file and sends each one to the agent server via `POST /run`. Collects rewards and writes the rollout output.
- **Claude Code Agent Server** — receives the task, seeds a session on the resources server to get a per-task MCP endpoint, then shells out to the `claude` CLI. Claude runs with `--output-format stream-json` so its tool calls and responses come back as structured events.
- **Workplace Assistant Resources Server** — the environment itself. It holds all five databases in memory per session, exposes them as MCP tools that Claude can call, and runs the verifier at the end to check whether the task was completed correctly.

---
## 4. What's Inside the Environment

### The Resources Server

The five databases live inside the **Workplace Assistant Resources Server** — a FastAPI server that the agent interacts with during a rollout. The server does three things:

1. **Seeds a per-task session** (`/seed_session`) — loads a fresh isolated copy of all five databases for each task, so concurrent rollouts don't interfere with each other
2. **Exposes the 27 tools** — the agent calls these tools (via MCP) to read and write the databases
3. **Runs the verifier** (`/verify`) — after the agent finishes, checks the final database state and returns the reward

State changes made by the agent (sending an email, updating a calendar event) are scoped to that session only.

### Toolkits

The 27 tools are organized into 6 toolkits. The agent has access to all of them on every task.

| Toolkit | Tools |
|---|---|
| **Company Directory** | `find_email_address` |
| **Email** | `get_email_information_by_id`, `search_emails`, `send_email`, `delete_email`, `forward_email`, `reply_email` |
| **Calendar** | `get_event_information_by_id`, `search_events`, `create_event`, `delete_event`, `update_event` |
| **Analytics** | `get_visitor_information_by_id`, `create_plot`, `total_visits_count`, `engaged_users_count`, `traffic_source_count`, `get_average_session_duration` |
| **Project Management** | `get_task_information_by_id`, `search_tasks`, `create_task`, `delete_task`, `update_task` |
| **Customer Relationship Manager** | `search_customers`, `update_customer`, `add_customer`, `delete_customer` |

### Databases

Each session gets its own isolated copy of five CSV-backed databases:

```
csv_data/processed/
├── emails.csv
├── calendar_events.csv
├── analytics_data.csv
├── project_tasks.csv
└── customer_relationship_manager.csv
```

### Verification

After the agent finishes, the verifier compares the **final state of the databases** against the expected outcome in `verifier_metadata.ground_truth`. It does not check which tools were called or in what order — only whether the database ended up in the right state. This is intentional: it gives the model freedom to find the correct answer any valid way.

Reward is **binary**: **1.0** if correct, **0.0** otherwise. It will not exceed 1.0 or fall between 0 and 1.

For multi-reward verification, visit: https://docs.nvidia.com/nemo/gym/main/build-verifiers/multi-reward-verification

---
## 5. How to Run It

### What is a config file and when do you need one?

NeMo Gym uses **YAML config files** to declare which servers to start and how to connect them. A config file answers: what servers exist, where do they live, and how should they talk to each other?

You need a config file whenever you run `gym env start` or `gym eval run` — it's how Gym knows what to start.

**What goes where:**

| File | Purpose | What belongs here |
|---|---|---|
| `env.yaml` | Secrets and environment-specific values | API keys, model URLs, model names, anything that changes per person or per cluster |
| `workplace_claude.yaml` | Environment wiring | Server declarations, which agent connects to which resources server, all non-secret config |

Keep `env.yaml` out of version control (it contains credentials). The wiring config can be committed.

**Where configs live:** By convention, environment configs live alongside their resources server:
```
resources_servers/workplace_assistant/configs/workplace_claude.yaml
```

### The config file

Create this file: ```resources_servers/workplace_assistant/configs/workplace_claude.yaml ``` with the contents below:
```yaml
head_server:                         # optional — defaults to host: 127.0.0.1, port: 11000
  host: 127.0.0.1
  port: 11001                        # only needed if 11000 is already taken

workplace_assistant:                 # instance name — what other servers reference with `name: workplace_assistant`
  resources_servers:                 # server type
    workplace_assistant:             # implementation name — tells Gym to look in resources_servers/workplace_assistant/
      entrypoint: app.py
      domain: agent                  # each task gets its own isolated copy of the databases;
                                     # without this, concurrent tasks would share state and corrupt each other
      expose_tools_over_mcp: true    # turns the 27 workplace tools into MCP tools for Claude Code

workplace_claude:                    # instance name — passed to `gym eval run --agent workplace_claude`
  responses_api_agents:              # server type
    claude_code_agent:               # implementation — looks in responses_api_agents/claude_code_agent/
      entrypoint: app.py
      resources_server:
        type: resources_servers
        name: workplace_assistant    # wires this agent to the resources server instance above
      model: ${anthropic_model_name}
      anthropic_api_key: ${anthropic_api_key}
      anthropic_base_url: ${anthropic_base_url}
```

The `${...}` placeholders are filled in from `env.yaml` at runtime — Gym merges the two files before starting.

```yaml
# env.yaml  (keep out of version control)
anthropic_api_key: <your-api-key>
anthropic_base_url: https://integrate.api.nvidia.com/v1
anthropic_model_name: nvidia/nemotron-3-ultra-550b-a55b
```

The valid fields inside each server block (e.g. `anthropic_api_key`, `model`, `concurrency`) are declared by that server's Pydantic config class in its `app.py`. Required fields have no default — omitting them causes a validation error on startup. Optional fields have defaults and only need to be set to override behavior.

### Start the servers

```bash
gym env start \
  --config env.yaml \
  --config resources_servers/workplace_assistant/configs/workplace_claude.yaml
```

**All `gym env start` flags:**

| Flag | Description |
|---|---|
| `--config PATH` | Config file to load. Repeatable — later files override earlier ones. |
| `--benchmark NAME` | Load a named benchmark config (shorthand for a pre-registered config path). |
| `--environment NAME` | Load a named environment config. |
| `--resources-server NAME` | Load a named resources-server config. |
| `--model-type NAME` | Load a named model-type config. |
| `--search-dir DIR` | Extra directory to search for named components. Repeatable. |
| `--model / -m` | Model name or checkpoint path. |
| `--model-url` | Model server base URL. |
| `--model-api-key` | Model server API key. |
| `-v` | Verbose logging (DEBUG level). |

### Run evaluation

```bash
gym eval run --no-serve \
  --config env.yaml \
  --config resources_servers/workplace_assistant/configs/workplace_claude.yaml \
  --agent workplace_claude \
  --input resources_servers/workplace_assistant/data/example.jsonl \
  --output results/workplace_claude_rollouts.jsonl \
  --limit 1
```

**All `gym eval run` flags:**

| Flag | Description |
|---|---|
| `--config PATH` | Config file to load. Repeatable. |
| `--agent / -a` | Which agent server (by config block name) to collect rollouts with. |
| `--input / -i` | Input tasks JSONL file. |
| `--output / -o` | Output rollouts JSONL file. |
| `--no-serve` | Skip starting servers — connect to already-running ones instead. |
| `--resume` | Resume from cached rollouts; re-run only tasks that haven't completed. |
| `--limit` | Stop after this many tasks. Useful for smoke tests. |
| `--num-repeats` | Number of rollouts per task (default 1). Use with `--resume` for pass@k evaluation. |
| `--concurrency` | Max number of tasks running in parallel. |
| `--split` | Dataset split to use: `train`, `validation`, or `benchmark`. |
| `--prompt-config` | YAML file with a prompt template to apply to inputs before sending. |
| `--temperature` | Sampling temperature (overrides the dataset value). |
| `--top-p` | Nucleus sampling top-p. |
| `--max-output-tokens` | Cap on output tokens per turn. |
| `--model / -m`, `--model-url`, `--model-api-key` | Model overrides (same as `gym env start`). |
| `--benchmark`, `--environment`, `--resources-server`, `--model-type`, `--search-dir` | Named config shorthands (same as `gym env start`). |
| `-v` | Verbose logging. |

### Output files

| File | Contents |
|---|---|
| `*_rollouts.jsonl` | Successful rollouts — one row per task, including full trajectory and reward. |
| `*_failures.jsonl` | Tasks that failed due to agent errors, timeouts, or malformed outputs. One row per attempt, with a `_ng_failure_class` field explaining the failure. These are retried automatically on `--resume` up to 3 times; tasks flagged `_ng_failure_terminal=True` are not retried. |
| `*_materialized_inputs.jsonl` | Resolved inputs that were sent to the agent (useful for debugging prompt issues). |
| `*_aggregate_metrics.json` | Summary stats: mean reward, mean tokens, turn counts. |

> **Note:** `_failures.jsonl` captures agent-level errors (the agent crashed, timed out, or returned an unparseable response). A task that completes but gets reward 0.0 is *not* a failure — it goes into `_rollouts.jsonl` with `reward: 0.0`.

A successful run on the example task should show `"mean/reward": 1.0`.

### Try it

**Step 1 — Start the servers**

This starts the Workplace Assistant Resources Server and the Claude Code Agent Server in the background. Wait a few seconds for them to come up before running eval.

In [ ]:
import subprocess, os

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))

proc = subprocess.Popen(
    ["gym", "env", "start",
     "--config", os.path.join(repo_root, "env.yaml"),
     "--config", os.path.join(repo_root, "resources_servers/workplace_assistant/configs/workplace_claude.yaml")],
    cwd=repo_root,
)
print(f"Servers starting (PID {proc.pid}) — wait ~15s before running the next cell")

You must wait until the servers are running before starting rollout collection. Wait for a message like this:


```text
[1] workplace_assistant (resources_servers/workplace_assistant)
{
    'config_path': 'workplace_assistant',
    'dir_path': 'path/to/resources_server',
    'entrypoint': 'app.py',
    'host': '127.0.0.1',
    'name': 'workplace_assistant',
    'pid': 5495,
    'port': 12370,
    'process_name': 'workplace_assistant',
    'server_type': 'resources_servers',
    'url': 'http://127.0.0.1:12370',
}
[2] workplace_claude (responses_api_agents/claude_code_agent)
{
    'config_path': 'workplace_claude',
...
    'url': 'http://127.0.0.1:12742',
}
```

**Step 2 — Run eval on the first task**

In [ ]:
import subprocess, os

# resolve paths relative to the repo root, not the notebook directory
repo_root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '../../..'))
output = os.path.join(repo_root, 'results/workplace_claude_rollouts.jsonl')
os.makedirs(os.path.dirname(output), exist_ok=True)

subprocess.run([
    "gym", "eval", "run", "--no-serve",
    "--config", os.path.join(repo_root, "env.yaml"),
    "--config", os.path.join(repo_root, "resources_servers/workplace_assistant/configs/workplace_claude.yaml"),
    "--agent", "workplace_claude",
    "--input", os.path.join(repo_root, "resources_servers/workplace_assistant/data/example.jsonl"),
    "--output", output,
    "--concurrency", "5",
], cwd=repo_root)


In [ ]:
import subprocess, os

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

subprocess.run(
    ["uv", "run", "gym", "eval", "prepare", "--benchmark", "ifeval"],
    cwd=repo_root,
    check=True,
)

100%|██████████| 1/1 [00:00<00:00,  5.01it/s]


Preparing benchmark: ifeval
Wrote 541 problems to /Users/artij/Projects/GymBrian/benchmarks/ifeval/data/ifeval_benchmark.jsonl
Benchmark data prepared at: /Users/artij/Projects/GymBrian/benchmarks/ifeval/data/ifeval_benchmark.jsonl


CompletedProcess(args=['uv', 'run', 'gym', 'eval', 'prepare', '--benchmark', 'ifeval'], returncode=0)

**Step 3 — Check the results**

In [55]:
import json, os

repo_root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '../../..'))
metrics_path = os.path.join(repo_root, 'results/workplace_claude_rollouts_aggregate_metrics.json')

with open(metrics_path) as f:
    print(json.dumps(json.load(f), indent=2))


[
  {
    "agent_ref": {
      "name": "workplace_claude"
    },
    "agent_metrics": {
      "mean/reward": 0.6,
      "mean/turns_used": 2.2,
      "mean/finished_naturally": 1.0,
      "mean/input_tokens": 46674.0,
      "mean/output_tokens": 819.2,
      "mean/total_tokens": 47493.2,
      "max/reward": 1.0,
      "max/turns_used": 3.0,
      "max/finished_naturally": 1.0,
      "max/input_tokens": 68921.0,
      "max/output_tokens": 1457.0,
      "max/total_tokens": 70378.0,
      "min/reward": 0.0,
      "min/turns_used": 1.0,
      "min/finished_naturally": 1.0,
      "min/input_tokens": 32278.0,
      "min/output_tokens": 331.0,
      "min/total_tokens": 32609.0,
      "median/reward": 1.0,
      "median/turns_used": 3.0,
      "median/finished_naturally": 1.0,
      "median/input_tokens": 42295.0,
      "median/output_tokens": 561.0,
      "median/total_tokens": 42800.0,
      "std/reward": 0.5477225575051662,
      "std/turns_used": 1.0954451150103321,
      "std/finished_nat

---
## 6. A Real Rollout

Here's what actually happened when we ran Nemotron on the first example task.

### The task

```
System: Today's date is Thursday, 2023-11-30 and the current time is 23:59:00.
        Meetings must not start before 9am or end after 6pm.

User:   Reply to carlos's last email about 'Task Update on Develop prototype
        for report generation' with 'Thanks for the update - I will get back
        to you tomorrow.'
```

The agent can't reply to an email without knowing its ID, so it needs to search first.

---

### Step 1 — Search for the email

**Tool call:** `email_search_emails`
```json
{"query": "Task Update on Develop prototype for report generation"}
```

**Result:** Returns a list of matching emails. Carlos's email has ID `00000057`.

```json
{"emails": [{"email_id": "00000057", "sender/recipient": "carlos.mendez@atlas.com",
              "subject": "Task Update on Develop prototype for report generation",
              "sent_datetime": "2023-11-29 09:12:00", ...}]}
```

---

### Step 2 — Reply to the email

**Tool call:** `email_reply_email`
```json
{"email_id": "00000057",
 "body": "Thanks for the update - I will get back to you tomorrow."}
```

**Result:**
```json
{"output": "Email replied successfully."}
```

---

### Final response

> *I've replied to Carlos's email (email_id: 00000057) with the message: "Thanks for the update - I will get back to you tomorrow."*

---

### Outcome

| Metric | Value |
|---|---|
| Model | `nvidia/nvidia/nemotron-3-ultra-nvfp4` |
| Tool calls | 2 |
| Input tokens | 22,960 |
| Output tokens | 348 |
| **Reward** | **1.0 ✓** |

**On tokens:** Input tokens count everything sent to the model on each turn — the system prompt, conversation history, all 27 tool schemas, and any tool results from prior steps. Because the full tool list is sent on every turn, input token counts are high even for short tasks. Output tokens count only what the model generates: its reasoning, tool call arguments, and final response text. The 22,960 input / 348 output split here is typical for a two-step task.

The verifier confirmed the reply was sent to the correct email with the correct body.

---
## 7. Swapping the Model or the Agent

The environment is a set of modular components. You can swap either one independently without touching the resources server or the dataset.

There are two separate things you might want to change:
- **The model** — which LLM is doing the reasoning (affects `env.yaml` or the model server block)
- **The agent harness** — the loop and tooling that runs around the model (affects which `responses_api_agents` entry you use)

---

### Changing the model (keeping Claude Code as the agent)

The Claude Code agent talks to any Anthropic-compatible endpoint via `anthropic_base_url`. Just swap `env.yaml` — no config file changes needed.

**build.nvidia.com API**
```yaml
anthropic_api_key: nvapi-...
anthropic_base_url: https://integrate.api.nvidia.com/v1
anthropic_model_name: nvidia/nemotron-3-ultra-550b-a55b
```

**Real Anthropic API**
```yaml
anthropic_api_key: sk-ant-...
anthropic_base_url: ""            # omit to use Anthropic's default endpoint
anthropic_model_name: claude-sonnet-4-6
```

**Local vLLM**
```bash
vllm serve meta-llama/Llama-3.1-8B-Instruct --port 8000
```
```yaml
anthropic_api_key: local          # any non-empty string
anthropic_base_url: http://localhost:8000
anthropic_model_name: meta-llama/Llama-3.1-8B-Instruct
```

**Gym-managed model server** (needed for training, since Gym needs to track token IDs):
```yaml
# Add to workplace_claude.yaml
policy_model:
  responses_api_models:
    vllm_model:
      entrypoint: app.py
      base_url: ${policy_base_url}
      api_key: ${policy_api_key}
      model: ${policy_model_name}

workplace_claude:
  responses_api_agents:
    claude_code_agent:
      ...
      model_server:              # reference the block above instead of anthropic_base_url
        type: responses_api_models
        name: policy_model
```

---

### Changing the agent harness

Swapping the agent harness means using a different `responses_api_agents` implementation. The resources server, dataset, and eval commands stay exactly the same — you only change the config file and the `--agent` flag.

**Key difference between agent types:** The Claude Code agent embeds its own inference client (the Anthropic SDK), so it only needs a URL and API key in the config. Most other agents — like `hermes_agent` — have no built-in client and always route inference through a Gym-managed model server. This means their config requires an explicit model server block, which wasn't needed for Claude Code.

**Example: switching to hermes_agent**

Create `workplace_hermes.yaml`:

```yaml
head_server:
  host: 127.0.0.1
  port: 11001

workplace_assistant:
  resources_servers:
    workplace_assistant:
      entrypoint: app.py              # same entrypoint — no change needed
      domain: agent
      expose_tools_over_mcp: false    # hermes_agent calls tools over plain HTTP, not MCP

policy_model:                          # required — hermes_agent has no built-in client
  responses_api_models:
    vllm_model:
      entrypoint: app.py
      base_url: ${policy_base_url}
      api_key: ${policy_api_key}
      model: ${policy_model_name}

workplace_hermes:
  responses_api_agents:
    hermes_agent:
      entrypoint: app.py
      resources_server:
        type: resources_servers
        name: workplace_assistant
      model_server:
        type: responses_api_models
        name: policy_model
      max_turns: 30
      temperature: 1.0
```

Then run with the new config:

```bash
gym env start \
  --config env.yaml \
  --config resources_servers/workplace_assistant/configs/workplace_hermes.yaml

gym eval run --no-serve \
  --config env.yaml \
  --config resources_servers/workplace_assistant/configs/workplace_hermes.yaml \
  --agent workplace_hermes \
  --input resources_servers/workplace_assistant/data/example.jsonl \
  --output results/workplace_hermes_rollouts.jsonl
```

---
## 8. Analyzing Results with BLADE

BLADE (Benchmark for LLM Agent Development and Evaluation) turns raw reward numbers into an actionable diagnostic report — identifying where the model fails, why, and what to do about it.

### The BLADE flow

BLADE analysis has two parts: **writing the report** and **validating it**.

The report itself is a markdown document that analyzes your rollouts: pass@1, task outcome buckets (always-pass / never-pass), dominant failure modes, and recommendations. It has to be written by a human or an AI reading the rollout file — `blade_toolkit` does not generate it automatically from rollouts.

To generate the report, use the [SKILL](https://github.com/NVIDIA-NeMo/Gym/blob/main/.claude/skills/nemo-gym-blade-analysis/SKILL.md).

To run with Claude Code:
```
/nemo-gym-blade-analysis Analyze the complete run in <results_dir>. Save the golden report to                 
<results_dir>workplace_claude_blade_report.md
```

In [ ]:
import subprocess, os

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
results_dir = os.path.join(repo_root, "results")

# Invoke the BLADE analysis skill via Claude Code CLI.
# It reads the rollouts, writes the golden report to results/.
result = subprocess.run(
    ["claude", "-p",
     f"/nemo-gym-blade-analysis Analyze the complete run in {results_dir}. "
     f"Save the golden report to {results_dir}/workplace_claude_blade_report.md"],
    cwd=repo_root, capture_output=True, text=True, timeout=1200
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print(result.stderr[-1000:])

### Generated Report Snapshot

#### Executive Summary

Nemotron Ultra achieved **52.48% pass@1** (286/545 tasks) on the Workplace Assistant benchmark. The benchmark covers five tool-using workplace categories (email, CRM, project management, calendar, analytics); email and CRM are the strongest categories at 67.9% and 65.1% respectively, while analytics is the weakest at 29.4%.

A single failure mode drives more than half of all failures: **the model systematically ignores the simulated date in the system prompt (2023-11-30) and uses its own training-time recency (2026-07-15)** when constructing date arguments for MCP tool calls. This date hallucination accounts for 142/259 failures (54.8%) and explains why calendar and analytics — the most date-sensitive categories — have the lowest pass rates. Fixing this one behavior is the highest-leverage intervention available.

The remaining failures break into behavioral issues (wrong argument choices, wrong tool selection, wrong conditional logic) and a small infrastructure tail of 429 rate-limit errors.

### Failure taxonomy

BLADE assigns each failed task a root-cause label:

| Label | Meaning | Typical fix |
|---|---|---|
| `KG` | Knowledge gap — model doesn't know the tool API or domain | Targeted SFT, better task docs |
| `UK` | Unreliable knowledge — sometimes passes, sometimes doesn't | RL, self-checking prompts |
| `BI` | Behavioral issue — capable but skips steps, gives up early, or loops | Shape the agent loop, reward intermediate verification |
| `TI` | Task/verifier issue — expected answer or timeout is wrong | Repair task or verifier, rerun baseline |
| `IR` | Infrastructure reliability — sandbox startup, network, scheduler | Fix infra, isolate flaky rows |

For the Workplace Assistant, common failure modes are `BI` (agent calls the wrong tool or skips the lookup step) and `KG` (model doesn't know which toolkit handles a given action).

### Reading the report

A BLADE report follows this structure:

```
## Executive Summary       ← key numbers and top finding in 3–5 sentences
## Artifact Inventory      ← rollout counts, coverage, repeat settings
## Aggregate Results       ← pass@1, pass@k, consistency by category
## Workflow Funnel         ← where in the tool-call chain tasks drop off
## Task Outcome Buckets    ← always-pass / sometimes-pass / never-pass split
## Dominant Failure Modes  ← top-k failure patterns with example trajectories
## Sometimes-Pass Dives    ← why the same task succeeds on some repeats and fails on others
## Recommendations         ← concrete next steps mapped to failure labels
```

The most diagnostic slice is **sometimes-pass tasks** — they reveal the conditions under which the model can succeed, which is exactly what you want to reinforce during training.

### Example: reading aggregate metrics

After running the single-task example above, `*_aggregate_metrics.json` will look like:

```json
{
  "mean/reward": 1.0,
  "mean/input_tokens": 22960,
  "mean/output_tokens": 348,
  "count": 1
}
```

At scale (all 1,260 tasks), you'd break `mean/reward` down by `verifier_metadata.category` to see which toolkit the model struggles with most — that category breakdown is your highest-leverage input for targeted improvement.

Once you have a report, `blade_toolkit` validates it with three steps:

| Step | Command | What it does |
|---|---|---|
| 1. Extract anchor facts | `extract-anchor-facts --golden <report.md>` | Pulls specific claims out of the report (pass rate, failure modes, etc.) |
| 2. Make shallow baseline | `make-shallow --input <report.md>` | Strips the report to just headings + aggregate tables — a negative control |
| 3. Calibrate | `calibrate --golden-report --anchor-facts --shallow-report` | Checks that the report has real diagnostic content beyond just numbers |

### What you need before running

- `*_rollouts.jsonl` from `gym eval run` — the raw rollout data you analyze
- A written BLADE report (markdown) — the analysis of those rollouts
- That's it. The three toolkit steps produce everything else themselves.

### Calibration output

```json
{
  "golden_vs_self": 0.99,   // report contains its own facts — should be ~1.0
  "shallow_vs_golden": 0.28, // shallow version misses the analysis — should be low
  "spread": 0.71             // gap between them — should be > 0.5
}
```

High spread means the report has real analytical content, not just aggregate numbers restated as prose. All three targets passed for this run.

In [ ]:
import subprocess, os

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
script = os.path.join(repo_root, ".claude/skills/nemo-gym-blade-analysis/scripts/blade_toolkit.py")
results_dir = os.path.join(repo_root, "results")

result = subprocess.run(
    ["uv", "run", "python", script, "extract-anchor-facts",
     "--golden",     os.path.join(results_dir, "workplace_claude_blade_report.md"),
     "--output",     os.path.join(results_dir, "workplace_claude_anchor_facts.json"),
     "--benchmark",  "workplace_assistant",
     "--model-name", "claude-code"],
    cwd=repo_root, capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr)


In [ ]:
import subprocess, os

repo_root = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
script = os.path.join(repo_root, '.claude/skills/nemo-gym-blade-analysis/scripts/blade_toolkit.py')
results_dir = os.path.join(repo_root, 'results')

result = subprocess.run(
    ['uv', 'run', 'python', script, 'make-shallow',
     '--input',  os.path.join(results_dir, 'workplace_claude_blade_report.md'),
     '--output', os.path.join(results_dir, 'workplace_claude_blade_shallow.md')],
    cwd=repo_root, capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

In [60]:
result = subprocess.run(
    ['uv', 'run', 'python', script, 'calibrate',
     '--golden-report',  os.path.join(results_dir, 'workplace_claude_blade_report.md'),
     '--anchor-facts',   os.path.join(results_dir, 'workplace_claude_anchor_facts.json'),
     '--shallow-report', os.path.join(results_dir, 'workplace_claude_blade_shallow.md'),
     '--output-dir',     results_dir],
    cwd=repo_root, capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

{
  "golden_vs_self": 1.0,
  "shallow_vs_golden": 0.3169,
  "spread": 0.6831,
  "targets": {
    "golden_vs_self_min": 0.85,
    "shallow_vs_golden_max": 0.4,
    "spread_min": 0.5
  },
  "note": "Deterministic public proxy. Treat failures as review signals, not official BLADE scores."
}


